# 12 — Shuffler & Chunker for Negative-Set Generation

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | [`03` SVM with kernel](03_svc_with_kernel.ipynb) | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | [`06` Weighted (WGKM) kernel](06_weighted_kernel.ipynb) | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | [`09` Spectrum encoder & NB](09_spectrum_encoder_and_differential.ipynb) | [`10` Gappy encoder](10_gappy_encoder.ipynb) | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | `**12**` Shuffler & chunker

`kmer.perturb` provides two background models for generating null-model sequences:

- **`KmerShuffler`** — preserves exact k-mer composition via random Eulerian paths in the De Bruijn graph.
- **`Chunker`** — block-level perturbation: split into chunks, optionally RC, shuffle, concatenate.

Both implement `BaseBackgroundModel` with `perturb()` / `perturb_many()`.

In [1]:
from collections import Counter
from kmer.perturb import KmerShuffler, Chunker, BaseBackgroundModel

## 1. KmerShuffler — k-mer composition preservation

In [2]:
seq = 'AAACCCGGGTTTAAACCCGGGTTT'
for k in [1, 2, 3]:
    sh = KmerShuffler(k=k, seed=42)
    out = sh.shuffle(seq)
    in_kmers = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
    out_kmers = Counter(out[i:i+k] for i in range(len(out) - k + 1))
    print(f'k={k}: in==out? {in_kmers == out_kmers}, out={out}')

k=1: in==out? True, out=TCGGTCCCATGCACAATAGTGTGA
k=2: in==out? True, out=AACCCGGTTAAAACCCGGGGTTTT
k=3: in==out? True, out=AAAACCCCGGGGTTAACCGGTTTT


## 2. Endpoint modes

In [3]:
seq = 'ACGTACGTACGTAAACCCGGGTTT'
for mode in ['preserve', 'free', 'crop']:
    sh = KmerShuffler(k=3, seed=42, endpoints=mode)
    out = sh.shuffle(seq)
    print(f'{mode:8s}: len={len(out):2d} out={out}')

preserve: len=24 out=ACGTACCCGGGTAAACGTACGTTT
free    : len=24 out=ACGTAAACGGGTACCCGTACGTTT
crop    : len=20 out=GTAAACGGGTACCCGTACGT


## 3. Chunker — block-level perturbation

In [4]:
ch = Chunker(min_size=2, max_size=4, seed=42)
seq = 'ACGTACGTACGTACGT'
out = ch.chunk(seq)
print(f'in : {seq}')
print(f'out: {out}')
print(f'len preserved: {len(out) == len(seq)}')
print(f'composition preserved: {Counter(seq) == Counter(out)}')

in : ACGTACGTACGTACGT
out: ACGCGTACGTTACGTA
len preserved: True
composition preserved: True


## 4. Chunker with strand flipping

In [5]:
ch = Chunker(min_size=2, max_size=4, flip_strand=True, seed=42)
out = ch.chunk('ACGTACGTACGTACGT')
print(f'with flip_strand: {out}')

with flip_strand: ACGTTAACGCGTTACG


## 5. Batch operations (parallel, reproducible)

In [6]:
seqs = ['ACGTACGTACGT', 'TTTTAAAACCCC', 'GCGCGCGCGCGC']
r1 = KmerShuffler(k=2, seed=42).shuffle_many(seqs, n_jobs=1)
r2 = KmerShuffler(k=2, seed=42).shuffle_many(seqs, n_jobs=4)
print(f'n_jobs=1 == n_jobs=4: {r1 == r2}')

n_jobs=1 == n_jobs=4: True
